# Bluetooth Low Energy — advertisement, GATT, notifikasi

Central memindai advertisement, tersambung ke periferal, lalu membaca, menulis, dan subscribe ke characteristic. Radio virtual di bawah berperilaku seperti yang sungguhan; ganti ke `o.UseNative()` untuk memakai adaptor Bluetooth Anda.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.19.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.19.0-preview.1"

## Urai advertisement secara manual
Advertising data adalah daftar struktur length–type–value; iBeacon adalah data pabrikan Apple.

In [ ]:
using IoTCom.Net.Transport.Ble;

var raw = Convert.FromHexString("020106" + "03030D18" + "040948524D" + "1AFF4C000215E2C56DB5DFFB48D2B060D0F5A71096E000070001C5");
var parsed = AdvertisingData.Parse("demo", raw, rssi: -68);
Console.WriteLine($"{parsed.Name} · services {string.Join(",", parsed.Services.Select(BleUuid.Name))} · iBeacon {parsed.IBeacon} · ≈{parsed.EstimatedDistance} m");
foreach (var f in AdvertisingData.Describe(raw)) Console.WriteLine($"  {f.Name,-5} {f.Value}");

## Pindai ruangan (virtual)

In [ ]:
var room = new VirtualBleNetwork();
room.AddHeartRateStrap();
room.AddEnvironmentSensor();
room.AddBeacon();
room.AddSmartPlug();
var physics = new VirtualBleSimulator(room);
var ble = BleCentral.Create(o => { o.UseVirtual(room); o.ReadOnly = true; });
await ble.ConnectAsync();
foreach (var ad in await ble.ScanAsync(TimeSpan.FromMilliseconds(600)))
    Console.WriteLine($"{ad.Name,-18} {ad.Rssi,4} dBm  {string.Join(", ", ad.Services.Select(BleUuid.Name))}{(ad.IBeacon is { } b ? $" iBeacon {b.Major}/{b.Minor}" : "")}");

## Sambung, baca, subscribe

In [ ]:
var strap = await ble.OpenAsync("C4:7C:8D:6A:21:0F");
foreach (var svc in strap.Services) Console.WriteLine($"{svc.Name}: {string.Join(", ", svc.Characteristics.Select(c => $"{c.Name} ({c.Properties})"))}");
Console.WriteLine($"battery {GattValue.Describe(BleUuid.Parse("2a19"), await strap.ReadAsync("2a19"))}");
using var beatWindow = new CancellationTokenSource(TimeSpan.FromSeconds(3));
_ = Task.Run(async () => { while (!beatWindow.IsCancellationRequested) { physics.Step(); await Task.Delay(250); } });
try { await foreach (var v in strap.SubscribeAsync(BleUuid.FromShort(0x2A37), beatWindow.Token)) Console.WriteLine(GattValue.Describe(BleUuid.FromShort(0x2A37), v)); }
catch (OperationCanceledException) { }

## Penulisan ditolak dalam mode read-only

In [ ]:
var plug = await ble.OpenAsync("D0:8E:3A:55:10:C2");
try { await plug.WriteAsync(VirtualBleNetwork.SmartPlugRelay, new byte[] { 1 }); }
catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }
await ble.DisposeAsync();

## Lebih lanjut
`iotcom ble scan` (radio sungguhan) atau `--sim`, `iotcom ble watch --sim C4:7C:8D:6A:21:0F 2a37`, *Perangkat Bluetooth di sekitar* di Gallery, dan sampel BleHeartRate. Lihat `docs/id/protocols/ble.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*